# 10 - Valutazione Finale sul Test Set
Addestramento di XGBoost con iperparametri ottimizzati su train+val e valutazione di XGBoost e GRU sul test set (Sezione 4.1 della tesi - RQ1).

In [ ]:
import sys, os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/t1dbg'
except ImportError:
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

## Import

In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
import xgboost as xgb
import keras
import json
from keras import layers
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    mean_absolute_percentage_error,
)

from lib.data import load_splits, rescale_data, calculate_metrics, print_results, HYPO, HYPER, L_BOUND, U_BOUND
from lib.tf_dnn import create_gru_model, predict_in_batches

Installazione di Optuna per il caricamento dello studio di ottimizzazione.

In [15]:
# installa optuna
!pip install optuna

## Funzioni di utilità
Funzioni per il caricamento dei migliori iperparametri, l'addestramento finale e la valutazione sul test set per XGBoost e GRU.

In [ ]:
def load_best_xgb_params(study_path="tuning/results/xgb_optuna_study.pkl"):
    """Carica i migliori iperparametri per XGB dallo studio di Optuna"""
    print("=" * 60)
    print("LOADING BEST XGBOOST HYPERPARAMETERS")
    print("=" * 60)

    if not os.path.exists(study_path):
        raise FileNotFoundError(f"XGBoost Optuna study not found at {study_path}")

    with open(study_path, "rb") as f:
        study = pickle.load(f)

    best_params = study.best_params.copy()
    best_mae = study.best_value

    print(f"Best validation MAE from tuning: {best_mae:.4f}")
    print(f"Best hyperparameters:")
    for param, value in best_params.items():
        if isinstance(value, float):
            print(f"  {param}: {value:.6f}")
        else:
            print(f"  {param}: {value}")

    # Aggiungi i parametri fissi per il training
    best_params.update({"random_state": 42, "device": "cuda:0"})

    return best_params, best_mae

In [ ]:
def train_final_xgb_model(train_set, val_set, X_cols, y_cols, best_params):
    """Addestra il modelo XGB sul set formato da train+val con i migliori iperparametri passati"""
    print("\n" + "=" * 60)
    print("TRAINING FINAL XGBOOST MODEL")
    print("=" * 60)

    # Combine train and validation sets
    combined_set = pd.concat([train_set, val_set], ignore_index=True)
    print(f"Combined training set size: {len(combined_set)}")
    print(f"  - Original train set: {len(train_set)}")
    print(f"  - Original val set: {len(val_set)}")

    # Create and train model
    model = xgb.XGBRegressor(**best_params)

    X_combined = combined_set[X_cols]
    y_combined = combined_set[y_cols[-1]]

    model.fit(X_combined, y_combined)

    return model

In [ ]:
def evaluate_xgb_on_test(model, test_set, X_cols, y_cols):
    """Valuta XGB sul test set."""
    print("\n" + "=" * 60)
    print("EVALUATING XGBOOST ON TEST SET")
    print("=" * 60)

    # Make predictions
    test_eval = test_set.copy()
    test_eval["y_pred"] = model.predict(test_eval[X_cols])

    # Prepare results
    test_eval = test_eval.rename(columns={y_cols[-1]: "target"})
    test_eval = rescale_data(test_eval, ["target", "y_pred"])

    # Select output columns
    output_columns = ["Timestamp", "Patient_ID", "bgClass", "target", "y_pred"]
    results = test_eval[output_columns]

    # Print results
    print("XGBoost Test Set Results:")
    print_results(results)

    return results

In [19]:
def save_xgb_model_and_results(
    model, results, models_path="models/test_set", outputs_path="outputs/test_set"
):
    """Salva il modello XGB otteuto e i risultati sul test set."""
    print(f"\nSaving XGBoost model and results...")

    # Create directories if they don't exist
    os.makedirs(models_path, exist_ok=True)
    os.makedirs(outputs_path, exist_ok=True)

    # Save model
    model_path = f"{models_path}/xgb.pickle"
    with open(model_path, "wb") as f:
        pickle.dump(model, f)
    print(f"XGBoost model saved to: {model_path}")

    # Save results
    results_path = f"{outputs_path}/xgb_output.csv"
    results.to_csv(results_path, index=False)
    print(f"XGBoost test results saved to: {results_path}")

In [ ]:
def load_gru_model_weights(weights_path="models/val_set/gru.weights.h5"):
    """Carica il modello GRU e i suoi pesi."""
    print("\n" + "=" * 60)
    print("LOADING GRU MODEL")
    print("=" * 60)

    if not os.path.exists(weights_path):
        raise FileNotFoundError(f"GRU weights not found at {weights_path}")

    # Create GRU model with same architecture as training
    model = create_gru_model()

    # Load weights
    model.load_weights(weights_path)
    print(f"GRU model weights loaded from: {weights_path}")

    # Print model summary
    print(f"\nGRU Model Architecture:")
    model.summary()

    return model

In [ ]:
def evaluate_gru_on_test(model, test_set, X_cols, y_cols):
    """Valuta GRU sul test set."""
    print("\n" + "=" * 60)
    print("EVALUATING GRU ON TEST SET")
    print("=" * 60)

    # Make predictions
    test_eval = test_set.copy()
    test_eval["y_pred"] = predict_in_batches(model, test_eval[X_cols], "gru").flatten()

    # Prepare results
    test_eval = test_eval.rename(columns={y_cols[-1]: "target"})
    test_eval = rescale_data(test_eval, ["target", "y_pred"])

    # Select output columns
    output_columns = ["Timestamp", "Patient_ID", "bgClass", "target", "y_pred"]
    results = test_eval[output_columns]

    # Print results
    print("GRU Test Set Results:")
    print_results(results)

    return results

In [22]:
def save_gru_results(results, outputs_path="outputs/test_set"):
    """Salva i risultati di GRU sul test-set"""
    print(f"\nSaving GRU results...")

    # Create directory if it doesn't exist
    os.makedirs(outputs_path, exist_ok=True)

    # Save results
    results_path = f"{outputs_path}/gru_output.csv"
    results.to_csv(results_path, index=False)
    print(f"GRU test results saved to: {results_path}")

## Pipeline di valutazione
XGBoost viene ri-addestrato su train+val con i parametri ottimali. GRU utilizza i pesi gia addestrati. Entrambi vengono valutati sul test set.

In [ ]:
print("FINAL TEST SET EVALUATION")
print("=" * 60)
print("Evaluating XGBoost (with optimal hyperparameters) and GRU on test set")
print("=" * 60)

In [ ]:
# Load data splits
train_set, val_set, test_set, X_cols, y_cols = load_splits()

print(f"Dataset sizes:")
print(f"  Train set: {len(train_set)} samples")
print(f"  Val set: {len(val_set)} samples")
print(f"  Test set: {len(test_set)} samples")
print(f"  Features: {len(X_cols)}")
print(f"  Target: {y_cols}")

In [25]:
# ===== VALUTAZONE XGBOOST =====

# Carica i migliori iperparametri ottenuti
best_params, best_val_mae = load_best_xgb_params()
# Addestra XGB
xgb_model = train_final_xgb_model(train_set, val_set, X_cols, y_cols, best_params)
# Valutalo sul test set
xgb_results = evaluate_xgb_on_test(xgb_model, test_set, X_cols, y_cols)
# Salva il modello e i risultati
save_xgb_model_and_results(xgb_model, xgb_results)

In [26]:
# ===== VALUTAZIONE GRU =====

# Carica il modello GRU
gru_model = load_gru_model_weights()
# Valutalo sul test set
gru_results = evaluate_gru_on_test(gru_model, test_set, X_cols, y_cols)
# Salva i risultati
save_gru_results(gru_results)

In [ ]:
print("\n" + "=" * 60)
print("EVALUATION COMPLETED.")
print("=" * 60)
print(f"Models saved in: models/test_set/")
print(f"Results saved in: outputs/test_set/")